# Simulação de Análise Estatística de Dados com NumPy Para a Área de Marketing

## 1. Definição do Problema de Negócio

**Contexto:** Uma plataforma de e-commerce coleta um volume significativo de dados sobre a interação dos usuários (visitas, duração da sessão, itens no carrinho e valor de compra). No entanto, as decisões sobre campanhas de marketing e UX atualmente são tomadas sem uma compreensão aprofundada desses padrões de comportamento.

**O Problema:** A falta de clareza sobre o que diferencia os clientes de alto valor dos visitantes casuais resulta em marketing genérico, desperdício de orçamento, perda de oportunidades de conversão e decisões não embasadas.

**Objetivo Principal:** Utilizar análise estatística dos dados de navegação e compra para segmentar clientes, identificar os principais indicadores de comportamento que levam à conversão e fornecer insights acionáveis para aumentar o ticket médio e a taxa de conversão.

**Perguntas-Chave da Análise:**
1. Qual é o perfil médio do usuário em termos de visitas, tempo de navegação e ticket médio?
2. Quais são os comportamentos distintos dos clientes de "Alto Valor"? 
3. Qual o comportamento de quem visita o site e não compra? Onde está a oportunidade?
4. Existe correlação entre tempo no site, itens no carrinho e valor da compra?

In [ ]:
# Importação das bibliotecas necessárias
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Define semente para reprodutibilidade dos resultados
np.random.seed(42)

## 2. Geração da Matriz de Dados

Para esta análise, simulei um conjunto de dados representando o comportamento de 500 usuários da plataforma. A base foi construída utilizando distribuições estatísticas da biblioteca `NumPy` para simular um cenário realista de e-commerce, onde o engajamento do usuário influencia as métricas de conversão.

Cada registro na matriz contém 4 métricas principais:

* **Visitas:** Número de vezes que o usuário visitou o site no mês;
* **Tempo no site:** Tempo total (em minutos) que o usuário passou navegando. *Usuários com mais visitas tendem a acumular mais tempo;*
* **Itens no carrinho:** Quantidade de produtos adicionados. *Influenciado positivamente pelo tempo de navegação;*
* **Valor da compra:** Valor total da compra finalizada (em R$). *Calculado com base em um ticket médio de R$ 35 por item, acrescido de uma variação aleatória para simular a realidade.*

In [ ]:
# Define o número de usuários para a simulação
num_usuarios = 500

# 1. Gera o número de visitas (valores inteiros aleatórios entre 1 e 50)
visitas = np.random.randint(low=1, high=51, size=num_usuarios)

# 2. Gera o tempo no site (distribuição normal, correlacionado com visitas)
# Média (loc) de 20 min, desvio padrão (scale) de 5, com um bônus por visita
tempo_no_site = np.random.normal(loc=20, scale=5, size=num_usuarios) + (visitas * 0.5)
tempo_no_site = np.round(tempo_no_site, 2)

# 3. Gera o número de itens no carrinho (dependente das visitas e do tempo)
itens_no_carrinho = np.random.randint(low=0, high=8, size=num_usuarios) + (visitas // 10)
itens_no_carrinho = (itens_no_carrinho + (tempo_no_site // 15)).astype(int)

# 4. Gera o valor da compra (correlacionado com os itens no carrinho)
# Preço médio por item de R$ 35, com uma variação aleatória baseada em distribuição normal
valor_compra = (itens_no_carrinho * 35) + np.random.normal(loc=0, scale=10, size=num_usuarios)

# Tratamento para garantir a integridade dos dados: 
# Compras não podem ser negativas e 0 itens no carrinho = R$ 0
valor_compra[itens_no_carrinho == 0] = 0
valor_compra[valor_compra < 0] = 0
valor_compra = np.round(valor_compra, 2)

# Unindo tudo em uma única matriz bidimensional (500 linhas x 4 colunas)
dados_ecommerce = np.column_stack((visitas, tempo_no_site, itens_no_carrinho, valor_compra))

# Exibindo as 5 primeiras linhas para conferência
print("Amostra da Matriz de Dados (Visitas | Tempo | Itens | Valor R$):")
print(dados_ecommerce[:5])

## 3. Análise Exploratória: Perfil Geral dos Usuários

Antes de realizar segmentações avançadas, precisei estabelecer uma "linha de base" (baseline). Para isso, extraí as estatísticas descritivas da matriz utilizando as funções matemáticas integradas do `NumPy`. 

O objetivo é descobrir o perfil médio do usuário da plataforma (média de visitas, tempo e itens) e entender a distribuição do faturamento (mediana, desvio padrão e valores extremos).

In [ ]:
# Calcula o perfil médio de todas as colunas de uma só vez
# O parâmetro axis=0 indica que queremos a média de cada coluna (verticalmente)
media_geral = np.mean(a=dados_ecommerce, axis=0)

# Calcula estatísticas específicas para a coluna de Valor da Compra (Índice 3)
mediana_valor = np.median(a=dados_ecommerce[:, 3])
desvio_padrao_valor = np.std(a=dados_ecommerce[:, 3])
valor_maximo = np.max(a=dados_ecommerce[:, 3])
valor_minimo = np.min(a=dados_ecommerce[:, 3])

print("--- Perfil Médio Geral do Usuário ---")
print(f"Visitas: {media_geral[0]:.0f}")
print(f"Tempo no site: {media_geral[1]:.1f} minutos")
print(f"Itens no carrinho: {media_geral[2]:.0f}")

print("\n--- Resumo Financeiro (Valor de Compra) ---")
print(f"Ticket Médio: R$ {media_geral[3]:.2f}")
print(f"Mediana: R$ {mediana_valor:.2f}")
print(f"Maior Compra: R$ {valor_maximo:.2f}")
print(f"Menor Compra: R$ {valor_minimo:.2f}")
print(f"Desvio Padrão: R$ {desvio_padrao_valor:.2f}")

## 4. Segmentação: Identificando Clientes de Alto Valor

Com a média geral estabelecida, é possível isolar o grupo de clientes que traz o maior retorno financeiro para a empresa. Considerei como "Alto Valor" os clientes cujo valor de compra final foi superior a R$ 250,00.

Utilizei a técnica de **indexação booleana** do `NumPy` para criar uma nova matriz contendo apenas os registros que atendem a essa condição, permitindo comparar o comportamento desse grupo VIP com a base geral.

In [ ]:
# Define a regra de negócio para o corte de alto valor
limite_alto_valor = 250.00

# Aplica a máscara booleana: retorna todas as colunas das linhas onde a coluna 3 é maior que 250
clientes_alto_valor = dados_ecommerce[dados_ecommerce[:, 3] > limite_alto_valor]

# O atributo .shape retorna uma tupla (linhas, colunas). Usei o índice 0 para saber a quantidade de clientes
total_clientes_vip = clientes_alto_valor.shape[0]
percentual_vip = (total_clientes_vip / num_usuarios) * 100

# Calcula1 as médias específicas deste novo segmento (axis=0)
medias_vip = np.mean(a=clientes_alto_valor, axis=0)

print(f"Total de Clientes de Alto Valor: {total_clientes_vip} ({percentual_vip:.1f}% da base)")

print("\n--- Perfil Médio do Cliente de Alto Valor ---")
print(f"Visitas: {medias_vip[0]:.0f}")
print(f"Tempo no site: {medias_vip[1]:.1f} minutos")
print(f"Itens no carrinho: {medias_vip[2]:.0f}")
print(f"Ticket Médio VIP: R$ {medias_vip[3]:.2f}")

## 5. Segmentação: Visitantes Engajados sem Compra

Além de entender quem traz mais receita, identifiquei a necessidade de analisar os usuários que visitam a plataforma, navegam, mas não convertem.

Para isso, filtrei os usuários cujo valor final de compra foi igual a zero. Meu objetivo aqui é entender se eles chegam a interagir com os produtos (adicionando ao carrinho) ou quanto tempo passam na plataforma antes de abandonar a sessão.

In [ ]:
# Aplica a máscara para focar apenas em quem não comprou nada
visitantes_sem_compra = dados_ecommerce[dados_ecommerce[:, 3] == 0]

total_sem_compra = visitantes_sem_compra.shape[0]
percentual_sem_compra = (total_sem_compra / num_usuarios) * 100

print(f"Total de Visitantes sem Compra: {total_sem_compra} ({percentual_sem_compra:.1f}% da base)")

# Se houver usuários nessa condição, é calculado o perfil médio deles para entender o abandono
if total_sem_compra > 0:
    medias_sem_compra = np.mean(a=visitantes_sem_compra, axis=0)
    print("\n--- Perfil Médio do Visitante sem Compra ---")
    print(f"Visitas: {medias_sem_compra[0]:.0f}")
    print(f"Tempo no site: {medias_sem_compra[1]:.1f} minutos")
    print(f"Itens no carrinho (abandonados): {medias_sem_compra[2]:.0f}")
else:
    print("\nNenhum usuário deixou de comprar nesta simulação.")

## 6. Validação Estatística: Correlação de Variáveis

Por fim, busquei validar estatisticamente as hipóteses de negócio. Existe uma correlação forte entre o tempo gasto no site e o valor final da compra? 

Utilizei a função `np.corrcoef` para calcular o coeficiente de correlação linear de Pearson entre as colunas da matriz. Valores próximos de 1 indicam uma forte correlação positiva (quando um sobe, o outro também sobe), enquanto valores próximos de 0 indicam ausência de relação linear.

In [ ]:
# Isolando as colunas em vetores unidimensionais para facilitar o cálculo cruzado
vetor_visitas = dados_ecommerce[:, 0]
vetor_tempo = dados_ecommerce[:, 1]
vetor_itens = dados_ecommerce[:, 2]
vetor_valor = dados_ecommerce[:, 3]

# Calculando as correlações de Pearson
# A função retorna uma matriz de correlação, o índice [0, 1] pega exatamente a intersecção entre as duas variáveis
corr_tempo_valor = np.corrcoef(x=vetor_tempo, y=vetor_valor)[0, 1]
corr_visitas_valor = np.corrcoef(x=vetor_visitas, y=vetor_valor)[0, 1]
corr_itens_valor = np.corrcoef(x=vetor_itens, y=vetor_valor)[0, 1]
corr_tempo_itens = np.corrcoef(x=vetor_tempo, y=vetor_itens)[0, 1]

print("--- Matriz de Correlação com Valor de Compra ---")
print(f"Tempo no Site x Valor da Compra: {corr_tempo_valor:.2f}")
print(f"Visitas x Valor da Compra: {corr_visitas_valor:.2f}")
print(f"Itens no Carrinho x Valor da Compra: {corr_itens_valor:.2f}")

print("\n--- Correlações Secundárias ---")
print(f"Tempo no Site x Itens no Carrinho: {corr_tempo_itens:.2f}")

## 7. Visualização de Dados

Para tornar a análise mais "palpável" e facilitar a comunicação com as partes interessadas (stakeholders), construí visualizações gráficas utilizando `Seaborn` e `Matplotlib`. 

O primeiro gráfico ilustra a relação direta entre o tempo de navegação e o valor final gasto, destacando visualmente a nossa segmentação de clientes de "Alto Valor". O segundo gráfico é um Mapa de Calor (Heatmap) que traduz a matriz de correlação matemática em um formato visual intuitivo.

In [ ]:
# Gráfico 1: Dispersão (Tempo no Site vs Valor da Compra)
plt.figure(figsize=(10, 6))

# Criando um array de categorias para colorir os pontos do gráfico
categorias_clientes = np.where(vetor_valor > limite_alto_valor, 'VIP (> R$ 250)', 'Regular')

cores_segmentos = {
    'VIP (> R$ 250)': '#2ca02c', # Verde para VIP
    'Regular': '#d62728'        # Vermelho para Regular
}

sns.scatterplot(
    x=vetor_tempo, 
    y=vetor_valor, 
    hue=categorias_clientes, 
    palette=cores_segmentos, 
    alpha=0.7,
    edgecolor=None
)

plt.title(label='Relação: Tempo no Site vs Valor da Compra', fontsize=14, fontweight='bold')
plt.xlabel(xlabel='Tempo de Navegação (minutos)', fontsize=12)
plt.ylabel(ylabel='Valor da Compra (R$)', fontsize=12)
plt.grid(visible=True, linestyle='--', alpha=0.5)
plt.legend(title='Segmento')
plt.show()

# Gráfico 2: Mapa de Calor das Correlações
plt.figure(figsize=(8, 6))

# Calculando a matriz de correlação completa das 4 variáveis
matriz_correlacao_completa = np.corrcoef(x=dados_ecommerce, rowvar=False)

# Nomes das colunas para os eixos do gráfico
nomes_variaveis = ['Visitas', 'Tempo no Site', 'Itens no Carrinho', 'Valor da Compra']

sns.heatmap(
    data=matriz_correlacao_completa, 
    annot=True, # Mostra os números dentro dos quadrados
    fmt='.2f', 
    cmap='Greens', 
    xticklabels=nomes_variaveis, 
    yticklabels=nomes_variaveis,
    vmin=-1,
    vmax=1,
    linewidths=0.5
)

plt.title(label='Mapa de Calor das Correlações Estatísticas', fontsize=14, fontweight='bold')
plt.show()

## 8. Relatório Executivo e Insights Acionáveis

Após a simulação e o processamento estatístico dos dados de navegação e compra, chegamos às seguintes conclusões e direcionamentos de negócio:

### 1. O Poder do Engajamento na Receita
Identifiquei que existe uma correlação linear positiva e expressiva entre o tempo que o usuário passa no site e o valor final da sua compra. Clientes que passam mais de 35 minutos navegando tendem a construir carrinhos significativamente maiores.
* **Ação Recomendada:** A equipe de UX/UI deve focar em estratégias de retenção de tela, como recomendações de produtos relevantes ("Quem comprou isso também levou..."), conteúdos interativos e vídeos demonstrativos nas páginas de produto para aumentar o tempo médio de sessão.

### 2. Segmentação Estratégica do Público VIP
Analisando o perfil isolado deste grupo, notei que eles representam uma parcela fundamental do faturamento e possuem uma frequência de visitas muito acima da média geral.
* **Ação Recomendada:** O time de Marketing deve alocar parte do orçamento para campanhas de fidelização e programas de recompensa (cashback, frete grátis anual) focados exclusivamente neste segmento, pois o Custo de Aquisição de Cliente (CAC) para mantê-los é menor do que o potencial de receita que eles geram.

### 3. Fricção na Construção do Carrinho
A correlação mais forte de todo o estudo foi entre a quantidade de itens no carrinho e o valor final. Parece óbvio, mas matematicamente me confirmou que cada item extra adicionado tem um peso enorme na previsibilidade de um ticket alto.
* **Ação Recomendada:** Implementar promoções de desconto progressivo ("Leve 3, pague 2") ou sugestões de combos (*cross-sell*) logo na página de checkout para incentivar a adição do "último item" antes do fechamento da compra.

### Conclusão
Com o uso da biblioteca `NumPy`, consegui ir além das métricas de vaidade. Substituí a intuição por dados quantitativos embasados, fornecendo um direcionamento claro para otimização de campanhas, redução de desperdício de orçamento e melhoria ativa da jornada do cliente. Esse projeto foi uma demonstração clara da relevância que a Ciência de Dados tem sobre a gestão eficaz de uma organização.